In [1]:
!pip install youtube-transcript-api


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import time
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable
)

In [3]:
df = pd.read_csv("cleaned_metadata.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (350, 6)


,video_id,title,publish_date,year,month,title_length
0,cw02dMpWStI,How to find good open source projects to contr...,2026-03-04 13:29:25+00:00,2026,3,76
1,0WjfKQdfeMU,NVIDIA-Certified Associate AI Infrastructure a...,2026-03-04 11:00:02+00:00,2026,3,88
2,ZldNCx4AvEM,Learn the basics of Git in 60 seconds with Bea...,2026-03-03 13:05:25+00:00,2026,3,54
3,E7o_WdfKszU,There's so much to learn - so how do you focus...,2026-03-02 13:17:28+00:00,2026,3,79
4,IBTx5aGj-6U,Build Your Own Video Sharing App – Loom Clone ...,2026-03-02 11:00:33+00:00,2026,3,86


In [4]:
if "video_id" in df.columns:
    print("video_id column exists ✅")
else:
    print("video_id column missing ❌")

video_id column exists ✅


In [5]:
df["video_id"] = df["video_id"].astype(str).str.strip()

In [6]:
video_id = df["video_id"].iloc[0]

api = YouTubeTranscriptApi()

try:
    transcript = api.fetch(video_id, languages=["en","hi"])

    print(transcript[0].text)

except Exception as e:
    print("Transcript not available:", e)

when I was in I was working I was


In [7]:
transcript_text = " ".join([seg.text for seg in transcript])

print(transcript_text[:500])

when I was in I was working I was probably working on one or two technologies and I was I was happy with what I was delivering but the scope of working on something else was always diminished right so what I can do I can probably go to like you know GitHub explored search all the public repository just search for react as the tag I have bunch of public projects which are like react based even react itself I found out some of them which are with high star having good readme maybe a community asso


In [8]:
api = YouTubeTranscriptApi()

def get_transcript(video_id):

    try:
        transcript = api.fetch(video_id, languages=["en","hi"])

        text = " ".join([seg.text for seg in transcript])

        return text

    except Exception:
        return None

In [ ]:
transcripts = []
failures = []

print("Extracting transcripts...\n")

for vid in df["video_id"]:

    text = get_transcript(vid)

    transcripts.append(text)

    if text is None:
        failures.append({
            "video_id": vid,
            "reason": "Transcript unavailable"
        })

    time.sleep(0.2)   

Extracting transcripts...



In [10]:
df["transcript"] = transcripts

df.head()

,video_id,title,publish_date,year,month,title_length,transcript
0,cw02dMpWStI,How to find good open source projects to contr...,2026-03-04 13:29:25+00:00,2026,3,76,when I was in I was working I was probably wor...
1,0WjfKQdfeMU,NVIDIA-Certified Associate AI Infrastructure a...,2026-03-04 11:00:02+00:00,2026,3,88,"Hey, this is Andrew Brown bringing you another..."
2,ZldNCx4AvEM,Learn the basics of Git in 60 seconds with Bea...,2026-03-03 13:05:25+00:00,2026,3,54,Let's learn the basics of Git in less than 60 ...
3,E7o_WdfKszU,There's so much to learn - so how do you focus...,2026-03-02 13:17:28+00:00,2026,3,79,And you know what one thing I will say is I th...
4,IBTx5aGj-6U,Build Your Own Video Sharing App – Loom Clone ...,2026-03-02 11:00:33+00:00,2026,3,86,"In this course, we're going to build a fully f..."


In [11]:
total = len(df)

available = df["transcript"].notnull().sum()

missing = df["transcript"].isnull().sum()

print("Total videos:", total)
print("Transcripts available:", available)
print("Missing transcripts:", missing)

coverage = (available / total) * 100

print("Transcript coverage:", round(coverage,2), "%")

Total videos: 350
Transcripts available: 345
Missing transcripts: 5
Transcript coverage: 98.57 %


In [12]:
fail_df = pd.DataFrame(failures)

fail_df.to_csv("transcript_failures.csv", index=False)

print("Failure log saved")

Failure log saved


In [13]:
df.to_csv("video_with_transcripts.csv", index=False)

print("Dataset saved as video_with_transcripts.csv")

Dataset saved as video_with_transcripts.csv
